# 🎙️ Kaggle: Whisper-Small (244M) High-Accuracy Multilingual STT Training
### Professional Fine-Tuning for Gujarati, Hindi & English Conversational Speech

This notebook upgrades the model to **`openai/whisper-small` (244M parameters)**
and trains on our curated **Pristine Conversational Dataset**.

---
### ⚡ IMPORTANT KAGGLE SETUP (Must do before running):
1. **Enable Internet**: Right sidebar ➔ **Notebook options** ➔ Toggle **Internet: ON**.
2. **Enable GPU**: Right sidebar ➔ **Accelerator** ➔ Select **GPU T4 x 2** or **GPU P100**.
3. **Upload Dataset**: Upload `conversational_stt_dataset.zip` via **+ Add Data** or upload directly into `/kaggle/working`.
4. **Run All Cells**: Click **Run All**! Training takes ~40-45 minutes on GPU.

## Step 1: Check GPU & Install Required Libraries

In [ ]:
# Verify NVIDIA GPU is active
!nvidia-smi

# Install speech and transformer packages
!pip install --timeout 60 --retries 5 -q transformers datasets accelerate soundfile librosa torchaudio
print('✅ Required libraries installed successfully!')

## Step 2: Locate & Extract Conversational Dataset

In [ ]:
import os, zipfile, glob, shutil

WORKING_DIR = '/kaggle/working'
dataset_dir = os.path.join(WORKING_DIR, 'dataset')
os.makedirs(dataset_dir, exist_ok=True)

# Look for zip file in /kaggle/input or /kaggle/working
zip_candidates = glob.glob('/kaggle/input/**/conversational_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/working/**/conversational_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/input/**/colab_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/working/**/colab_stt_dataset*.zip', recursive=True) + \
                 glob.glob('/kaggle/input/**/*.zip', recursive=True)

target_zip = None
for z in zip_candidates:
    if 'stt' in z.lower() or 'dataset' in z.lower() or 'conversational' in z.lower():
        target_zip = z
        break

if target_zip and os.path.exists(target_zip):
    print(f'📦 Found dataset zip at: {target_zip}')
    print('Extracting files...')
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(dataset_dir)
    print('✅ Extraction complete!')
else:
    print('Checking if files are already extracted...')

# Find manifest.csv
manifest_candidates = glob.glob(f'{dataset_dir}/**/manifest*.csv', recursive=True) + \
                      glob.glob('/kaggle/input/**/manifest*.csv', recursive=True)

if not manifest_candidates:
    raise FileNotFoundError('❌ manifest.csv not found! Please upload conversational_stt_dataset.zip')

manifest_path = manifest_candidates[0]
manifest_dir = os.path.dirname(manifest_path)
print(f'✅ Found manifest at: {manifest_path}')
print(f'Dataset root directory: {manifest_dir}')

## Step 3: Load & Verify Dataset Manifest

In [ ]:
import pandas as pd

df = pd.read_csv(manifest_path)
print(f'Total samples in dataset: {len(df):,}')
print(df.head(5)[['audio_path', 'transcript', 'language', 'duration']])

# Resolve absolute audio paths
def resolve_audio_path(p):
    p_str = str(p).strip()
    if os.path.exists(p_str):
        return p_str
    fname = os.path.basename(p_str)
    c1 = os.path.join(manifest_dir, 'wav_chunks', fname)
    if os.path.exists(c1):
        return c1
    c2 = os.path.join(manifest_dir, fname)
    if os.path.exists(c2):
        return c2
    matches = glob.glob(f'{manifest_dir}/**/{fname}', recursive=True)
    if matches:
        return matches[0]
    return p_str

df['resolved_path'] = df['audio_path'].apply(resolve_audio_path)
valid_df = df[df['resolved_path'].apply(os.path.exists)].copy()
print(f'✅ Valid audio clips ready for training: {len(valid_df):,} / {len(df):,}')

print('\nBreakdown by Language:')
for lang, count in valid_df['language'].value_counts().items():
    dur = valid_df[valid_df['language'] == lang]['duration'].sum()
    print(f'  - {lang.upper()}: {count:,} clips ({dur/3600:.2f} Hours)')

## Step 4: Load Whisper-Small Processor & Define Dataset Loader

In [ ]:
import torch
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from torch.utils.data import Dataset

MODEL_NAME = 'openai/whisper-small'
print(f'Loading {MODEL_NAME} Processor...')
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

class STTConversationalDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = row['resolved_path']
        transcript = str(row['transcript']).strip()

        audio_data, sr = sf.read(audio_path)
        if audio_data.ndim > 1:
            audio_data = audio_data.mean(axis=1)

        input_features = self.processor.feature_extractor(
            audio_data, sampling_rate=16000, return_tensors='pt'
        ).input_features[0]

        labels = self.processor.tokenizer(transcript).input_ids
        return {'input_features': input_features, 'labels': labels}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')

        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')

        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch['labels'] = labels
        return batch

# 90% Train / 10% Validation Split
train_df = valid_df.sample(frac=0.90, random_state=42)
val_df = valid_df.drop(train_df.index)

train_dataset = STTConversationalDataset(train_df, processor)
val_dataset = STTConversationalDataset(val_df, processor)
data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.bos_token_id
)
print(f'Train samples: {len(train_dataset):,} | Validation samples: {len(val_dataset):,}')

## Step 5: Initialize Whisper-Small Model (244M Params)

In [ ]:
print(f'🚀 Initializing {MODEL_NAME} with 244M parameters...')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Model successfully loaded! Total parameters: {total_params:,}')

## Step 6: Professional Fine-Tuning Pipeline (Gentle LR: 1.5e-5, Cosine Warmup, 8 Epochs)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, WhisperForConditionalGeneration

# Defensive check: re-initialize model if kernel restarted
if 'model' not in globals() or model is None:
    MODEL_NAME = 'openai/whisper-small'
    print(f'Loading {MODEL_NAME} for training...')
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    model.config.use_cache = False

OUTPUT_MODEL_DIR = '/kaggle/working/whisper_small_finetuned'

# Hyperparameters tailored to prevent catastrophic forgetting
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_MODEL_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch size = 16
    learning_rate=1.5e-5,           # Gentle and stable
    warmup_steps=150,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    num_train_epochs=8,
    gradient_checkpointing=True,    # Fits 244M in 16GB VRAM
    fp16=True,                      # Fast GPU acceleration
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_eval_batch_size=8,
    predict_with_generate=False,    # Fast validation loss tracking
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model='loss',
    greater_is_better=False,
    save_total_limit=2,
    report_to=['tensorboard'],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print('🚀 Starting Training on Kaggle GPU...')
trainer.train()
print('🎉 Training Complete!')

## Step 7: Export Best Model & Create 1-Click Download ZIP

In [ ]:
# Save best model and processor
FINAL_SAVE_DIR = '/kaggle/working/finetuned_whisper_small'
print(f'Saving final model to: {FINAL_SAVE_DIR}...')
trainer.save_model(FINAL_SAVE_DIR)
processor.save_pretrained(FINAL_SAVE_DIR)

# Package into 1-click download zip
ZIP_OUTPUT = '/kaggle/working/finetuned_whisper_small.zip'
print(f'Packaging into {ZIP_OUTPUT}...')
with zipfile.ZipFile(ZIP_OUTPUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FINAL_SAVE_DIR):
        for file in files:
            fpath = os.path.join(root, file)
            arcname = os.path.relpath(fpath, FINAL_SAVE_DIR)
            zf.write(fpath, arcname)

zip_size_mb = os.path.getsize(ZIP_OUTPUT) / (1024 * 1024)
print(f'\n🎉 SUCCESS! Fine-tuned Whisper-Small Model is ready!')
print(f'📁 Download Location: {ZIP_OUTPUT} ({zip_size_mb:.2f} MB)')
print('👉 In Kaggle: Look at the right sidebar under Output section ➔ Click the 3 dots on finetuned_whisper_small.zip ➔ Click Download!')